<a href="https://colab.research.google.com/github/ayushpaliwal1920/Deep_Learning/blob/main/35_age_gender(Functional_api).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [19]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
utkface-new.zip: Skipping, found more recently modified local copy (use --force to force download)


In [20]:
import zipfile
zip = zipfile.ZipFile('/content/utkface-new.zip' , 'r')
zip.extractall('/content')
zip.close()


In [21]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [22]:
folder_path ='/content/utkface_aligned_cropped/UTKFace'

In [23]:
age = []
gender = []
img_path = []

for file in os.listdir(folder_path):
    if file.endswith(".jpg"):
        age.append(int(file.split('_')[0]))
        gender.append(int(file.split('_')[1]))
        img_path.append(file)

In [24]:
len(age)

23708

In [25]:
df = pd.DataFrame({'age' : age , 'gender' : gender , 'img' : img_path})

In [26]:
df.shape

(23708, 3)

In [27]:
df.head()

,age,gender,img
0,30,1,30_1_1_20170112215057720.jpg.chip.jpg
1,26,0,26_0_2_20170117172558276.jpg.chip.jpg
2,25,1,25_1_0_20170116205342932.jpg.chip.jpg
3,1,0,1_0_2_20161219160747326.jpg.chip.jpg
4,55,0,55_0_3_20170119201235270.jpg.chip.jpg


In [28]:
train_df = df.sample(frac = 1 , random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state = 0).iloc[20000:]

In [29]:
train_df.shape

(20000, 3)

In [30]:
test_df.shape

(3708, 3)

In [31]:
#  data Aug :

train_datagen = ImageDataGenerator(
     rescale = 1./255,
     rotation_range = 30,
     width_shift_range = 0.2,
     height_shift_range = 0.2,
     shear_range = 0.2,
     zoom_range = 0.2,
     horizontal_flip = True
)

test_datagen = ImageDataGenerator(rescale = 1./255)

In [58]:
def custom_generator(generator):
    while True:
        X, y = next(generator)

        yield X, {
            "age": y[0],
            "gender": y[1]
        }

In [59]:
train_gen = custom_generator(train_generator)
test_gen = custom_generator(test_generator)

In [60]:
# train_generator = train_datagen.flow_from_dataframe(
#        train_df,
#        directory = folder_path,
#        x_col = 'img',
#        y_col = ['age','gender'],
#        target_size = (200,200),
#        class_mode = 'multi_output')

# test_generator = test_datagen.flow_from_dataframe(test_df,
#                                                        directory = folder_path,
#                                                        x_col = 'img',
#                                                        y_col = ['age','gender'],
#                                                        target_size = (200,200),
#                                                        class_mode = 'multi_output'
#                                                        )

In [61]:
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

In [62]:
resNet = ResNet50(include_top=False , input_shape=(200,200,3))

In [63]:
resNet.trainable = False

output = resNet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512,activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear' , name = 'age')(dense3)
output2 = Dense(1,activation='sigmoid' , name = 'gender')(dense4)



In [64]:
model = Model(inputs = resNet.input , outputs = [output1,output2])

In [65]:
model.compile(
    optimizer='adam',
    loss={
        'age': 'mae',
        'gender': 'binary_crossentropy'
    },
    metrics={
        'age': 'mae',
        'gender': 'accuracy'
    },
    loss_weights={
        'age': 1,
        'gender': 99
    }
)

In [66]:
model.fit(
    train_gen,
    epochs=10,
    validation_data=test_gen,
    steps_per_epoch=len(train_generator),
    validation_steps=len(test_generator)
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 249s 375ms/step - age_loss: 15.4148 - age_mae: 15.4148 - gender_accuracy: 0.5102 - gender_loss: 0.9424 - loss: 108.7138 - val_age_loss: 15.5611 - val_age_mae: 15.5628 - val_gender_accuracy: 0.5202 - val_gender_loss: 0.6924 - val_loss: 84.1065
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 228s 366ms/step - age_loss: 15.1393 - age_mae: 15.1393 - gender_accuracy: 0.5231 - gender_loss: 0.6922 - loss: 83.6661 - val_age_loss: 14.9043 - val_age_mae: 14.9043 - val_gender_accuracy: 0.5221 - val_gender_loss: 0.6922 - val_loss: 83.4317
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 225s 360ms/step - age_loss: 14.9494 - age_mae: 14.9494 - gender_accuracy: 0.5228 - gender_loss: 0.6967 - loss: 83.9188 - val_age_loss: 14.4078 - val_age_mae: 14.4078 - val_gender_accuracy: 0.5224 - val_gender_loss: 0.6922 - val_loss: 82.9383
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 224s 359ms/step - age_loss: 14.8159 - age_mae: 14.8159 - gender_accuracy: 0.5230 - gender_loss: 0.6923 - loss: 83